In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/clean_provider_data.csv")

print("Clean dataset loaded successfully")
print(df.shape)

Clean dataset loaded successfully
(1296739, 81)


In [3]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 rndrng_npi
2 rndrng_prvdr_last_org_name
3 rndrng_prvdr_first_name
4 rndrng_prvdr_mi
5 rndrng_prvdr_crdntls
6 rndrng_prvdr_ent_cd
7 rndrng_prvdr_st1
8 rndrng_prvdr_st2
9 rndrng_prvdr_city
10 rndrng_prvdr_state_abrvtn
11 rndrng_prvdr_state_fips
12 rndrng_prvdr_zip5
13 rndrng_prvdr_ruca
14 rndrng_prvdr_ruca_desc
15 rndrng_prvdr_cntry
16 rndrng_prvdr_type
17 rndrng_prvdr_mdcr_prtcptg_ind
18 tot_hcpcs_cds
19 tot_benes
20 tot_srvcs
21 tot_sbmtd_chrg
22 tot_mdcr_alowd_amt
23 tot_mdcr_pymt_amt
24 tot_mdcr_stdzd_amt
25 drug_sprsn_ind
26 drug_tot_hcpcs_cds
27 drug_tot_benes
28 drug_tot_srvcs
29 drug_sbmtd_chrg
30 drug_mdcr_alowd_amt
31 drug_mdcr_pymt_amt
32 drug_mdcr_stdzd_amt
33 med_sprsn_ind
34 med_tot_hcpcs_cds
35 med_tot_benes
36 med_tot_srvcs
37 med_sbmtd_chrg
38 med_mdcr_alowd_amt
39 med_mdcr_pymt_amt
40 med_mdcr_stdzd_amt
41 bene_avg_age
42 bene_age_lt_65_cnt
43 bene_age_65_74_cnt
44 bene_age_75_84_cnt
45 bene_age_gt_84_cnt
46 bene_feml_cnt
47 bene_male_cnt
48 bene_race_wht_cnt
49 bene_

In [4]:
state_summary = (
    df.groupby("rndrng_prvdr_state_abrvtn")
      .agg(
          total_providers=("rndrng_npi", "nunique"),
          total_beneficiaries=("tot_benes", "sum"),
          total_services=("tot_srvcs", "sum"),
          total_submitted_charges=("tot_sbmtd_chrg", "sum"),
          total_medicare_payments=("tot_mdcr_pymt_amt", "sum"),
          avg_medicare_payment=("tot_mdcr_pymt_amt", "mean")
      )
      .reset_index()
      .sort_values("total_medicare_payments", ascending=False)
)

state_summary.head(10)

,rndrng_prvdr_state_abrvtn,total_providers,total_beneficiaries,total_services,total_submitted_charges,total_medicare_payments,avg_medicare_payment
8,CA,105363,34155154,380605643.2,5.996005e+10,1.551556e+10,147258.156220
13,FL,85872,29221303,374407372.1,4.523734e+10,1.200349e+10,139783.534162
51,TX,86803,24434786,279760168.5,4.147035e+10,9.690709e+09,111640.256607
41,NY,88642,23050569,196314881.6,3.467162e+10,7.426551e+09,83781.405137
20,IL,51368,15282421,144170478.7,2.103441e+10,4.872599e+09,94856.707916
38,NJ,39694,13968752,155759980.8,2.198186e+10,4.779671e+09,120412.926894
45,PA,63361,17019840,158774442.5,1.763815e+10,4.691632e+09,74046.046099
7,AZ,26694,8494698,97730456.1,1.431753e+10,3.637294e+09,136258.871273
26,MD,26679,9337389,107518810.4,1.265835e+10,3.400895e+09,127474.599849
34,NC,44655,12811611,109608734.1,1.374158e+10,3.263886e+09,73091.165042


In [5]:
specialty_summary = (
    df.groupby("rndrng_prvdr_type")
      .agg(
          total_providers=("rndrng_npi", "nunique"),
          total_beneficiaries=("tot_benes", "sum"),
          total_services=("tot_srvcs", "sum"),
          total_submitted_charges=("tot_sbmtd_chrg", "sum"),
          total_medicare_payments=("tot_mdcr_pymt_amt", "sum"),
          avg_medicare_payment=("tot_mdcr_pymt_amt", "mean")
      )
      .reset_index()
      .sort_values("total_medicare_payments", ascending=False)
)

specialty_summary.head(15)

,rndrng_prvdr_type,total_providers,total_beneficiaries,total_services,total_submitted_charges,total_medicare_payments,avg_medicare_payment
66,Nurse Practitioner,203212,28889475,238137691.2,3.058725e+10,1.015083e+10,4.995191e+04
69,Ophthalmology,17519,10631801,154357570.7,2.708892e+10,8.245419e+09,4.706558e+05
46,Internal Medicine,93627,22848147,198029391.9,2.526678e+10,7.918245e+09,8.457224e+04
19,Clinical Laboratory,2743,24260562,304775826.6,2.460164e+10,6.928002e+09,2.525702e+06
38,Hematology-Oncology,9158,3028158,400609720.3,2.531456e+10,6.874966e+09,7.507061e+05
6,Ambulatory Surgical Center,5667,3696501,49942356.1,3.647656e+10,6.013483e+09,1.061140e+06
29,Family Practice,83639,16566776,129359858.1,1.553267e+10,5.265333e+09,6.295308e+04
5,Ambulance Service Provider,9324,5593163,115249869.6,2.087220e+10,4.454183e+09,4.777116e+05
25,Diagnostic Radiology,32843,57441175,218015183.1,2.775237e+10,4.113341e+09,1.252425e+05
11,Cardiology,19764,18142596,63125622.1,1.523649e+10,3.699997e+09,1.872089e+05


In [6]:
top_payment_providers = (
    df[
        [
            "rndrng_npi",
            "rndrng_prvdr_last_org_name",
            "rndrng_prvdr_first_name",
            "rndrng_prvdr_state_abrvtn",
            "rndrng_prvdr_type",
            "tot_benes",
            "tot_srvcs",
            "tot_sbmtd_chrg",
            "tot_mdcr_pymt_amt"
        ]
    ]
    .sort_values("tot_mdcr_pymt_amt", ascending=False)
)

top_payment_providers.head(25)

,rndrng_npi,rndrng_prvdr_last_org_name,rndrng_prvdr_first_name,rndrng_prvdr_state_abrvtn,rndrng_prvdr_type,tot_benes,tot_srvcs,tot_sbmtd_chrg,tot_mdcr_pymt_amt
812603,1629407069,"Exact Sciences Laboratories, Llc",Unknown,WI,Clinical Laboratory,614503,614504.0,4.184772e+08,3.063578e+08
689528,1538144910,Laboratory Corporation Of America Holdings,Unknown,NC,Clinical Laboratory,1560327,16587932.0,1.286033e+09,2.244011e+08
234434,1184045619,"Guardant Health, Inc.",Unknown,CA,Clinical Laboratory,38402,46254.0,4.037111e+08,2.096292e+08
722501,1558672279,"Natera, Inc.",Unknown,CA,Clinical Laboratory,29417,56156.0,2.737185e+08,1.955762e+08
82854,1063497451,Laboratory Corporation Of America Holdings,Unknown,NJ,Clinical Laboratory,1029624,13820423.0,1.035530e+09,1.584273e+08
970217,1740733708,"Tempus Ai, Inc.",Unknown,IL,Clinical Laboratory,28798,89595.0,2.017730e+08,1.479997e+08
840,1003053851,Jeng,Aaron,CA,Internal Medicine,839,120972.0,1.737540e+08,1.230430e+08
888219,1689607384,Caredx Inc.,Unknown,CA,Clinical Laboratory,15026,40611.0,1.726212e+08,1.205544e+08
1207354,1932145778,Quest Diagnostics Incorporated,Unknown,NJ,Clinical Laboratory,778459,9856188.6,9.984699e+08,1.181674e+08
1161896,1891731626,Quest Diagnostics Clinical Laboratories Inc,Unknown,FL,Clinical Laboratory,691918,9246143.0,9.627712e+08,1.133153e+08


In [7]:
os.makedirs("../data/processed/hive_outputs", exist_ok=True)

state_summary.to_csv("../data/processed/hive_outputs/state_summary.csv", index=False)
specialty_summary.to_csv("../data/processed/hive_outputs/specialty_summary.csv", index=False)
top_payment_providers.to_csv("../data/processed/hive_outputs/top_payment_providers.csv", index=False)

print("Hive-style analytics outputs saved successfully.")

Hive-style analytics outputs saved successfully.
